# Regresión Logística
## Integrantes:
- Martínez Marcelo Ingrid Aylen
- Pérez Evaristo Eris
- Ramírez Venegas Alexa Paola

Para esta práctica generaremos y entrenaremos modelos de regresión logística, perceptrones y regresión lineal.

1. Genera los nodos de una gráfica computacional basada en una super clase

In [12]:
import numpy as np
class Node:
  """ Clase nodo """
  def __call__ (self, x):
    return self.forward(x)

  def __str__(self):
    return str(self.out)

2. El nodo principal para los tres modelos sera el nodo de PreActivation que define la funcion de preactivación $wx + b$

In [13]:

class PreActivacion(Node):
  """ Clase del nodo principal de los tres modelos que define la función de preactivación
  Author:
    Martínez Marcelo Ingrid Aylen
    Pérez Evaristo Eris
    Ramírez Venegas Alexa Paola
  """
  def __init__(self, input_size: int, output_size: int):
    """ Constructor del nodo de preactivación
    Args:
      input_size (int): tamaño de la entrada
      output_size (int): tamaño de la salida
    """
    self.w = np.random.randn(input_size, output_size) * 0.1
    self.b = np.zeros((1, output_size))

  def forward(self, x: np.array):
    """ Función forward del nodo de preactivación
    Args:
      x (array): entrada
    Returns:
      Nodo de preactivación
    """
    self.x = x
    self.out = np.dot(x, self.w) + self.b
    return self

  def backward(self, consumer_grad, lr=0.1):
    """ Función backward del nodo de preactivación
    Args:
      consumer_grad (array): gradiente del nodo posterior
      lr (float): tasa de aprendizaje
    Returns:
      Gradiente del nodo de preactivación
    """
    m = self.x.shape[0]                          # número de muestras
    dw = np.dot(self.x.T, consumer_grad) / m     # gradiente respecto a W
    db = np.mean(consumer_grad, axis=0, keepdims=True)  # gradiente respecto a b

    # actualización de parámetros
    self.w -= lr * dw
    self.b -= lr * db

    # gradiente que se pasa hacia atrás (para capas previas a PreActivacion)
    return np.dot(consumer_grad, self.w.T)


3. Define el nodo de la funcion Sigmoide para la regresión logística.

In [14]:
class NodoSigmoide(Node):
  """ Clase nodo sigmoide para regresión logística
  Author:
    Martínez Marcelo Ingrid Aylen
    Pérez Evaristo Eris
    Ramírez Venegas Alexa Paola
  """
  def forward(self, x):
    """ Función foward del nodo sigmoide
    Returns:
      Nodo sigmoide
    """
    self.prev = x
    self.out = 1.0 / (1.0 + np.exp(-x.out))
    return self

  def backward(self, consumer_grade):
    """ Función backward del nodo sigmoide
    Returns:
      calculo del gradiente de la función sigmoide
    """
    local_grad = self.out * (1 - self.out)
    return self.prev.backward(consumer_grade * local_grad)

4. Define el nodo para la función de entropía cruzada binaria cuya función es $-yln(f) + (1-y)ln(1-f)$

$y = etiqueta real$
$f = predecida$

In [15]:
class NodoEntropiaCruzadaB(Node):
  """ Clase del nodo Entropía Cruzada para regresión logística
  Author:
    Martínez Marcelo Ingrid Aylen
    Pérez Evaristo Eris
    Ramírez Venegas Alexa Paola
  """
  def __call__(self, x, y):
    """ Función call definida para ser llamada al usar el nodo para la función de pérdida
    Args:
      x (array): predicción
      y (array): etiqueta real
    Returns:
      Nodo de entropía cruzada
    """
    self.prev, self.y = x, y.reshape(-1,1)
    eps = 1e-9
    self.out = - np.mean(self.y * np.log(x.out + eps) + (1 - self.y) * np.log(1 - x.out + eps))
    return self

  def backward(self):
    """ Función backward del nodo de entrpía cruzada
    Returns:
      Gradiente del nodo de entropía cruzada
    """
    m = self.y.shape[0]
    grad = (self.prev.out - self.y) / m
    return self.prev.backward(grad)

5. Entrena el modelo de regresión logística usando datos a partir de sklearn:


Utiliza 100 épocas y una tasa de aprendizaje de 0.1



In [25]:
from matplotlib.pylab import RandomState
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

class ModeloRegresionLogistica:

  """Clasificador Logistic Regression

    Parameters
    ------------
    eta : float
      Learning rate (entre 0.0 y 1.0)
    epochs : int
      Numero de pasos para entrenamiento.


    Attributes
    -----------
    errors : list
      Número de clases incorrectamente clasificadas para cada epoch
    fun_Preactivación: Nodo
      Función de preactivación para el modelo de regresión logística
    sigmoide : Nodo
      Función sigmoide para el modelo de regresión logística
    fun_perdida : Nodo
      Función de pérdida para el modelo de regresión logística
  """
  def __init__(self, eta=0.1, epochs=100, verbose=False):
    self.eta = eta
    self.epochs = epochs
    self.errors = []
    self.verbose = verbose
    self.fun_Preactivacion = PreActivacion(2,1)
    self.sigmoide = NodoSigmoide()
    self.fun_perdida = NodoEntropiaCruzadaB()

  def forward(self, X: np.array):
    """Función de preactivación
    Args:
      X (Array) : Datos de entrada
    Returns:
      Nodo sigmoide dada la función de preactivación
    """
    z = self.fun_Preactivacion(X)
    a = self.sigmoide(z)
    return a

  def train(self, X, y):
    """ Función de entrenamiento del modelo de regresión logística
    Args:
      X (Array) : Datos de entrada
      y (array) : Etiquetas
    Returns:
      Modelo entrenado
    """
    for t in range(self.epochs):
      a = self.forward(X)
      loss = self.fun_perdida(a, y).out
      self.fun_perdida.backward()
      self.errors.append(loss)
    return self


  def activation(self, z: np.array):
    """Función de activación sigmoide
    Args:
      z (Array) : Datos de entrada
    Returns:
      Valor calculado de la función sigmode
    """
    return self.sigmoide(z)

  def predict(self, X):
    """ Función de predicción
    Args:
      X (Array) : Datos a predecir
    Returns:
      Predicción de los datos, si es mayor a 0.5 es 1, en caso contrario 0.
    """
    a = self.forward(X).out
    return (a >= 0.5).astype(int)


# Entrenamiento para Regresión Logistica
# Datos
# n redundant para que no se rompa el codigo (debe ser <= n features)
# Random state para que la semilla sea la misma y se puedan comprobar que los resultados son los mismos con el perceptrón
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, n_informative=2, random_state=10)

# División
x_train, x_eval, y_train, y_eval = train_test_split(X, y,test_size=0.3, random_state=42)

lr = ModeloRegresionLogistica(eta=0.1, epochs=100, verbose=True)
lr.train(x_train, y_train)

6. Evalúa el resultado usando el reporte de clasificación

In [26]:
# Evaluar con el reporte de clasificación sklearn
yPrediccionRegLog = lr.predict(x_eval)
print("\nReporte de clasificación - Modelo1: Regresión Logística:")
print(classification_report(y_eval, yPrediccionRegLog))
print("Pesos finales:", lr.fun_Preactivacion.w)
print("Sesgo final:", lr.fun_Preactivacion.b)


Reporte de clasificación - Modelo1: Regresión Logística:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89       149
           1       0.90      0.88      0.89       151

    accuracy                           0.89       300
   macro avg       0.89      0.89      0.89       300
weighted avg       0.89      0.89      0.89       300

Pesos finales: [[ 0.06077368]
 [-0.00179395]]
Sesgo final: [[-5.18478072e-06]]


7. Define un nodo computacional para la función escalonada del perceptrón y comprueba que la clasificación es la misma que con regresión logística evaluando el resultado.

In [27]:
class NodoEscalonada(Node):
  """ Clase nodo con función escalonada
  Author:
    Martínez Marcelo Ingrid Aylen
    Pérez Evaristo Eris
    Ramírez Venegas Alexa Paola
  """
  def forward(self, x):
    """ Función forward del nodo escalonado
    Args:
      x (array) : entrada
    Returns:
      Nodo de función escalonada. Si es mayor a 0 es 1, caso contrario es 0.
    """
    self.prev = x
    # Activación escalonada (0 ó 1)
    self.out = (x.out >= 0).astype(int)
    return self

  def backward(self, consumer_grade):
    """ Función backward para el nodo escalonado
    Args:
      consumer_grade (array): gradiente del nodo posterior
    Returns:
      calculo del gradiente de la función escalonada
    """
    # El perceptrón clásico no usa derivadas,
    # Podemos pasar el gradiente sin modificar
    return self.prev.backward(consumer_grade)
    # o gradiente cero
    # return self.prev.backward(np.zeros_like(self.prev.out))

Para probar que clasifica igual que la regresión logística:

In [28]:
# Usamos los pesos aprendidos por la regresión logística
perceptron_act = NodoEscalonada()
yPrediccionPerceptron = perceptron_act(lr.fun_Preactivacion(x_eval)).out

print("\nReporte de clasificación - Modelo2: Perceptrón:")
print(classification_report(y_eval, yPrediccionPerceptron))


Reporte de clasificación - Modelo2: Perceptrón:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89       149
           1       0.90      0.88      0.89       151

    accuracy                           0.89       300
   macro avg       0.89      0.89      0.89       300
weighted avg       0.89      0.89      0.89       300



8. Define un nodo para la función objetivo del error cuadrático y entrena un modelo de regresión lineal $f(x) = wx + b$ con los datos:



```
  from sklearn.datasets import make_regression

  x. y = make_regression(n_samples=1000, n_features=2, n_informative=2)
  x_train, x_eval, y_train, y_eval = train_test_split(x, y, test_size=0.3)
```

Evalúa usando mean_squared_error y r2_score de sklearn.

In [29]:
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error, r2_score
class NodoMSE(Node):
  """ Clase nodo con función del error cuadrático medio
  Author:
    Martínez Marcelo Ingrid Aylen
    Pérez Evaristo Eris
    Ramírez Venegas Alexa Paola
  """
  def __call__(self, x, y):
    """ Función call definida para ser llamada una vez se invoque el nodo
    Args:
      x (array): predicción
      y (array): etiqueta real
    Returns:
      Nodo de error cuadrático medio
    """
    # Guardamos la predicción (x) y el valor real (y)
    self.prev, self.y = x, y.reshape(-1, 1)
    # Cálculo del MSE
    self.out = np.mean((self.prev.out - self.y) ** 2)
    return self

  def backward(self):
    """ Función backward para el nodo de error cuadrático medio
    Returns:
      Gradiente del nodo de error cuadrático medio respecto a la salida
    """
    # Gradiente de MSE respecto a la salida
    m = self.y.shape[0]
    grad = 2 * (self.prev.out - self.y) / m
    return self.prev.backward(grad)

In [30]:
class ModeloRegresionLineal:
  """Clasificador Logistic Regression

    Parameters
    ------------
    eta : float
      Learning rate (entre 0.0 y 1.0)
    epochs : int
      Numero de pasos para entrenamiento.


    Attributes
    -----------
    errors : list
      Número de clases incorrectamente clasificadas para cada epoch
    fun_Preactivación: Nodo
      Función de preactivación para el modelo de regresión lineal
    fun_perdida : Nodo
      Función de pérdida para el modelo de regresión lineal
  """
  def __init__(self, eta=0.1, epochs=100):
    """ Incialización del modelo de regresión lineal
    Args:
      eta (float): tasa de aprendizaje
      epochs (int): número de épocas
    """
    self.eta = eta
    self.epochs = epochs
    self.errors = []
    self.fun_Preactivacion = PreActivacion(2, 1)  # f(x) = wx + b
    self.fun_perdida = NodoMSE()

  def forward(self, X: np.array):
    """Función de preactivación
    Args:
      X (Array) : Datos de entrada
    Returns:
      Nodo de preactivación con los datos de entrada
    """
    return self.fun_Preactivacion(X)

  def train(self, X, y):
    """ Función de entrenamiento del modelo de regresión lineal
    Args:
      X (Array) : Datos de entrada
      y (Array) : etiquetas
    Returns:
      Modelo entrenado con la función de preactivación y el error cuadrático medio
    """
    for t in range(self.epochs):
      y_pred = self.forward(X)
      loss = self.fun_perdida(y_pred, y).out
      self.fun_perdida.backward()
      self.errors.append(loss)
    return self

  def predict(self, X):
    """ Función de predicción
    Args:
      X (Array) : Datos a predecir
    Returns:
      Predicción de los datos
    """
    return self.forward(X).out

In [31]:
# Datos regresión
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2)

# División en train/test
x_train, x_eval, y_train, y_eval = train_test_split(X, y, test_size=0.3)

# Entrenamiento
rl = ModeloRegresionLineal(eta=0.1, epochs=100)
rl.train(x_train, y_train)

# Predicciones
y_pred_eval = rl.predict(x_eval)

#Evaluación
print("\nReporte Modelo3: Regresión Lineal")
print("MSE:", mean_squared_error(y_eval, y_pred_eval))
print("R2:", r2_score(y_eval, y_pred_eval))
print("Pesos finales:", rl.fun_Preactivacion.w)
print("Sesgo final:", rl.fun_Preactivacion.b)


Reporte Modelo3: Regresión Lineal
MSE: 3160.415871876424
R2: 0.06123361053307219
Pesos finales: [[1.61065672]
 [0.96278673]]
Sesgo final: [[-0.06214462]]


9. Recuerda que cada nodo debe de tener una función forward que computa la función y otra backward que computa el gradiente. Los pesos que se actualizarán serán los de pre-activación:


```
  pre.w -= lr.a.grad
  pre.b -= lr.a.grad_b
```
NOTA: Ya se implementó este ejercicio